In [ ]:
import pandas as pd

# Load the CSV file
file_path = '/mnt/data/july_backtest.csv'
data = pd.read_csv(file_path)

# Convert the 'Datetime' column to datetime objects
data['Datetime'] = pd.to_datetime(data['Datetime'])

# Add a new column with the day of the week
data['day_of_week'] = data['Datetime'].dt.day_name()

# Calculate the count of result=1 and result=-1 for each day of the week
result_counts = data.groupby(['day_of_week', 'Result']).size().unstack(fill_value=0)


# Save the result counts to CSV
result_counts_file = '/mnt/data/result_counts_by_day_of_week.csv'

result_counts.to_csv(result_counts_file)

import matplotlib.pyplot as plt

# Plotting the result counts by day of the week
result_counts.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Result Counts by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Counts')
plt.legend(title='Result')
plt.tight_layout()
plt.show()


In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Directory containing the CSV files
directory_path = 'E:\Signal Backtesting\Output\Monthly backtest with previous optimization'

# Initialize an empty DataFrame to store combined results
combined_result_counts = pd.DataFrame()

# Loop through all CSV files in the directory
for filename in os.listdir(directory_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(directory_path, filename)
        
        # Load the CSV file
        data = pd.read_csv(file_path)

        # Convert the 'Datetime' column to datetime objects
        data['Datetime'] = pd.to_datetime(data['Datetime'])

        # Add a new column with the day of the week
        data['day_of_week'] = data['Datetime'].dt.day_name()

        # Calculate the count of result=1 and result=-1 for each day of the week
        result_counts = data.groupby(['day_of_week', 'Result']).size().unstack(fill_value=0)

        # Combine the results
        combined_result_counts = combined_result_counts.add(result_counts, fill_value=0)

# Save the combined result counts to CSV
result_counts_file = 'E:\Signal Backtesting\Output\combined_result_counts_by_day_of_week.csv'
combined_result_counts.to_csv(result_counts_file)


In [14]:
import plotly.graph_objects as go
# Plotting the combined result counts by day of the week using Plotly
fig = go.Figure()

for result in combined_result_counts.columns:
    fig.add_trace(go.Bar(
        x=combined_result_counts.index,
        y=combined_result_counts[result],
        name=f'Result {result}'
    ))

fig.update_layout(
    title='Combined Result Counts by Day of the Week',
    xaxis_title='Day of the Week',
    yaxis_title='Counts',
    barmode='stack'
)

fig.show()

In [12]:
# Directory containing the CSV files
directory_path = 'E:\Signal Backtesting\Output\Monthly backtest with previous optimization'  # Change this to your local directory path

# Initialize an empty DataFrame to store combined results
combined_result_counts = pd.DataFrame()

# List of original backtest files to process
backtest_files = [
    'Apr_backtest_with_Mar.csv',
    'Feb-backtest_with_Jan.csv',
    'July_backtst_with_June.csv',
    'June_backtest_with_May.csv',
    'Mar_backtest_with_Feb.csv',
    'May_backtest_with_Apr.csv'
]

# Loop through the original backtest CSV files in the directory
for filename in backtest_files:
    file_path = os.path.join(directory_path, filename)
    
    # Load the CSV file
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue
    
    # Ensure the 'Datetime' column exists
    if 'Datetime' not in data.columns:
        print(f"'Datetime' column missing in {filename}")
        continue

    # Try to parse the 'Datetime' column
    try:
        data['Datetime'] = pd.to_datetime(data['Datetime'], errors='coerce')
    except Exception as e:
        print(f"Error parsing 'Datetime' column in {filename}: {e}")
        continue

    # Drop rows with NaT in 'Datetime'
    data = data.dropna(subset=['Datetime'])

    # Add new columns for month and day of the week
    data['month'] = data['Datetime'].dt.to_period('M')
    data['day_of_week'] = data['Datetime'].dt.day_name()

    # Calculate the count of result=1 and result=-1 for each day of the week and each month
    monthly_result_counts = data.groupby(['month', 'day_of_week', 'Result']).size().unstack(fill_value=0)

    # Combine the results using outer join
    if combined_result_counts.empty:
        combined_result_counts = monthly_result_counts
    else:
        combined_result_counts = combined_result_counts.add(monthly_result_counts, fill_value=0)

# Ensure combined_result_counts has appropriate indexes
combined_result_counts = combined_result_counts.reset_index().set_index(['month', 'day_of_week'])

# Save the combined result counts to CSV
combined_result_counts_file = 'combined_result_counts_by_day_of_week.csv'  # Change this to your desired output directory
combined_result_counts.to_csv(combined_result_counts_file)
